<a href="https://colab.research.google.com/github/tomvangasse/ATLAS-top-tagging-ML-project/blob/main/notebooks/04_robustness.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Robustness Test

Both the XGboost and MLP models were trained on nominal simulation. This is our best guess of reality.


The systematic-variation files contain the same kinds of jets as nominal, but simulated with one thing deliberately changed. For example:
- the detector's energy scale shifted
- the energy resolution made worse
- a different parton-shower model used


Robustness is asking:
> 'If reality is slightly different from the nominal simulation, how much might each tagger's background rejection change?'



A small change means the model is robust, and can be trusted more with real world data.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

!rm -rf ATLAS-top-tagging-ML-project
!git clone https://github.com/tomvangasse/ATLAS-top-tagging-ML-project.git
import sys
sys.path.insert(0, "/content/ATLAS-top-tagging-ML-project")

Mounted at /content/drive
Cloning into 'ATLAS-top-tagging-ML-project'...
remote: Enumerating objects: 76, done.
remote: Counting objects: 100% (76/76), done.
remote: Compressing objects: 100% (68/68), done.
remote: Total 76 (delta 23), reused 20 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (76/76), 931.89 KiB | 13.31 MiB/s, done.
Resolving deltas: 100% (23/23), done.


In [2]:
!pip install -q cernopendata-client
!cernopendata-client get-file-locations --recid 80030 --protocol http \
    | sed 's#.*/##' | sed -E 's/_[0-9]+\.h5\.gz//' | sort | uniq -c


Aborted!


Using these variations for the robustness test:
1) esup
    - Energy scale shifted up
2) esdown
    - Energy scale shifted down
3) cer
    - Energy resolution made worse
4) angular
    - Uses Herwig’s angular-ordered shower, instead of Pythia, which changes how the jet’s internal structure is simulated


---



Downloading and unzipping these may take a while

In [ ]:
%cd /content/drive/MyDrive/cern_data
import os
!pip install -q cernopendata-client
VARS = ["esup", "esdown", "cer", "angular"]
for v in VARS:
    if os.path.exists(f"{v}_000.h5"):      # already downloaded: skip
        print("Already have", f"{v}_000.h5")
        continue
    urls = !cernopendata-client get-file-locations --recid 80030 --protocol http | grep "/{v}_" | head -1
    print("Downloading", urls[0])
    !wget -q -c {urls[0]}
!gunzip -f *.h5.gz 2>/dev/null || true
!ls -lh
%cd /content

## Rebuild the models

The models are build empty, and the saved weights are loaded into them

In [18]:
from google.colab import drive; drive.mount('/content/drive')
!rm -rf ATLAS-top-tagging-ML-project && git clone https://github.com/tomvangasse/ATLAS-top-tagging-ML-project.git
import sys; sys.path.append("ATLAS-top-tagging-ML-project")

import numpy as np, pandas as pd, h5py, torch, joblib, xgboost as xgb
from src.models import TopTaggerMLP
from src.metrics import background_rejection
from src.data import load, FEATURES, LABEL, WEIGHT
DATA = "/content/drive/MyDrive/cern_data/"
OUT = "/content/drive/MyDrive/cern_outputs/"
device = "cuda" if torch.cuda.is_available() else "cpu"

xgb_model = xgb.XGBClassifier(); xgb_model.load_model(DATA + "xgb_model.json")

mlp = TopTaggerMLP(n_features=15)
mlp.load_state_dict(torch.load(DATA + "mlp_model.pt", map_location=device))
mlp.to(device).eval()

scaler = joblib.load(DATA + "mlp_scaler.joblib")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Cloning into 'ATLAS-top-tagging-ML-project'...
remote: Enumerating objects: 76, done.
remote: Counting objects: 100% (76/76), done.
remote: Compressing objects: 100% (68/68), done.
remote: Total 76 (delta 23), reused 20 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (76/76), 931.89 KiB | 13.31 MiB/s, done.
Resolving deltas: 100% (23/23), done.


Function for scoring models.

It takes any DataFrame of jets and returns the scores of each model in a dictionary.

In [19]:
def score_both(df):
    X = df[FEATURES]
    s_xgb = xgb_model.predict_proba(X)[:, 1]
    X_s = torch.tensor(scaler.transform(X).astype(np.float32)).to(device)
    with torch.no_grad():
        s_mlp = torch.sigmoid(mlp(X_s)).cpu().numpy()
    return {"XGBoost": s_xgb, "MLP": s_mlp}

Check for files that only have 1 class, they need both backgroud and signal labels.

`y_nom == 1` is a mask selecting the nominal top jets. `y_nom == 0` selects the nominal background.

In [22]:
nominal = load(DATA + "test_nominal_000.h5")
nom_scores = score_both(nominal)

def check(df, name):
    counts = df[LABEL].value_counts().to_dict()
    print(f"{name}: {len(df)} jets, label counts {counts}")

def rejection_for(var_df, eff=0.5):
    var_scores = score_both(var_df)
    labels_var = var_df[LABEL].values
    has_sig, has_bkg = (labels_var == 1).any(), (labels_var == 0).any()
    y_nom = nominal[LABEL].values

    out = {}
    for name in ["XGBoost", "MLP"]:
        s_var, s_nom = var_scores[name], nom_scores[name]

        if has_sig and has_bkg:
            y, s = labels_var, s_var

        elif has_bkg:       # varied background + nominal signal
            y = np.concatenate([y_nom[y_nom == 1], labels_var])
            s = np.concatenate([s_nom[y_nom == 1], s_var])

        else:               # varied signal + nominal background
            y = np.concatenate([labels_var, y_nom[y_nom == 0]])
            s = np.concatenate([s_var, s_nom[y_nom == 0]])

        out[name] = background_rejection(y, s, eff)
    return out

## Building the results table

For each working point (50% and 80%) it computes the nominal rejection, then the rejection on each variation.

In [37]:
VARIATIONS = {
    "nominal": None,
    "angular": "angular_000.h5",
    "cer": "cer_000.h5",
    "esdown": "esdown_000.h5",
    "esup": "esup_000.h5"
}

rows = []
for eff in (0.5, 0.8):
    nom = {n: background_rejection(nominal[LABEL].values, s, eff)
           for n, s in nom_scores.items()}
    for var, fname in VARIATIONS.items():
        r = nom if fname is None else rejection_for(load(DATA + fname, n=500_000), eff)
        for model_name in r:
            rows.append({"eff": eff, "variation": var, "model": model_name,
                         "rejection": r[model_name],
                         "change_%": 100 * (r[model_name] - nom[model_name]) / nom[model_name]})

results = pd.DataFrame(rows)
print(results.round(2).to_string(index=False))
results.to_csv(OUT + "robustness.csv", index=False)

 eff variation   model  rejection  change_%
 0.5   nominal XGBoost      36.26      0.00
 0.5   nominal     MLP      37.75      0.00
 0.5   angular XGBoost      40.19     10.85
 0.5   angular     MLP      39.87      5.63
 0.5       cer XGBoost      38.36      5.81
 0.5       cer     MLP      37.88      0.36
 0.5    esdown XGBoost      38.49      6.15
 0.5    esdown     MLP      37.85      0.29
 0.5      esup XGBoost      38.15      5.21
 0.5      esup     MLP      37.73     -0.04
 0.8   nominal XGBoost       8.64      0.00
 0.8   nominal     MLP       9.00      0.00
 0.8   angular XGBoost       9.22      6.72
 0.8   angular     MLP       9.43      4.85
 0.8       cer XGBoost       8.59     -0.60
 0.8       cer     MLP       8.99     -0.04
 0.8    esdown XGBoost       8.60     -0.43
 0.8    esdown     MLP       9.00      0.02
 0.8      esup XGBoost       8.56     -0.99
 0.8      esup     MLP       8.96     -0.41


## Corrected comparisons

The table above compares every variation with `test_nominal_000`, but the variation files contain **different jets** from that file. So part of each change is just because the jets are different, not because of the variation. The next cells check which files share jets, then compare each variation with a fair reference:

- **esup, esdown and cer** contain the same collisions as each other, so they are compared with **each other**:
    - The energy-scale effect is half the difference between esup and esdown.
    - The resolution effect is cer compared with the average of esup and esdown.
- **Control:** `test_nominal_001` is compared with `test_nominal_000`. Both are nominal, so any change is just sample-to-sample noise.
- **Shower model (angular)** shares no jets with any other file, so it is compared with the **average of the two nominal files**.

The results are saved to `robustness_corrected.csv`.

### Which files contain the same jets?

For each pair of files, the fraction of jets in the first that also appear in the second (matched by event number).

In [30]:
EV = "EventInfo_mcEventNumber"   # event number column, used only to match jets between files
TEST_FILE = "test_nominal_000.h5"

evs = {n: load(DATA + (TEST_FILE if f is None else f), n=500_000)[EV].values
       for n, f in VARIATIONS.items()}
for a in evs:
    print(f"{a:8s}", {b: f"{np.isin(evs[a], evs[b]).mean():.0%}" for b in evs})

nominal  {'nominal': '100%', 'angular': '0%', 'cer': '3%', 'esdown': '3%', 'esup': '3%'}
angular  {'nominal': '0%', 'angular': '100%', 'cer': '0%', 'esdown': '0%', 'esup': '0%'}
cer      {'nominal': '3%', 'angular': '0%', 'cer': '100%', 'esdown': '100%', 'esup': '100%'}
esdown   {'nominal': '3%', 'angular': '0%', 'cer': '100%', 'esdown': '100%', 'esup': '100%'}
esup     {'nominal': '3%', 'angular': '0%', 'cer': '100%', 'esdown': '100%', 'esup': '100%'}


### Control test

How much does rejection change between two different nominal test files? This is the size of the noise from using a different sample of jets.

In [ ]:
import os, re
if os.path.exists(DATA + "test_nominal_001.h5"):
    print("Already have test_nominal_001.h5")
else:
    %cd /content
    all_urls = !cernopendata-client get-file-locations --recid 80030 --protocol http
    clean = [re.sub(r"\x1b\[[0-9;]*m", "", u).strip() for u in all_urls]   # strip hidden colour codes
    url = [u for u in clean if "/test_nominal_001" in u][0]
    !wget -c --progress=bar:force:noscroll "{url}"
    !gunzip -f /content/test_nominal_001.h5.gz
    !cp -v /content/test_nominal_001.h5 {DATA}

Testing the new test data with my results

In [ ]:
rej = results.set_index(["eff", "variation", "model"])["rejection"]
control = load(DATA + "test_nominal_001.h5", n=500_000)
r_ctrl = {}
for eff in (0.5, 0.8):
    r_ctrl[eff] = rejection_for(control, eff)
    for m in ["XGBoost", "MLP"]:
        r_nom = rej[eff, "nominal", m]
        print(f"{eff:.0%} {m:8s} nominal {r_nom:.2f}  control {r_ctrl[eff][m]:.2f}  "
              f"change {100 * (r_ctrl[eff][m] - r_nom) / r_nom:+.2f}%")

### Corrected results table

In [ ]:
rows = []
for eff in (0.5, 0.8):
    for m in ["XGBoost", "MLP"]:
        up, dn, cer = rej[eff, "esup", m], rej[eff, "esdown", m], rej[eff, "cer", m]
        ref = (up + dn) / 2                              # reference for the detector files
        n0, n1 = rej[eff, "nominal", m], r_ctrl[eff][m]
        nom_avg = (n0 + n1) / 2                          # reference for angular
        ang = rej[eff, "angular", m]
        rows += [
            {"eff": eff, "model": m, "variation": "energy scale",     "change_%": 100 * abs(up - dn) / 2 / ref},
            {"eff": eff, "model": m, "variation": "resolution",       "change_%": 100 * (cer - ref) / ref},
            {"eff": eff, "model": m, "variation": "control (nominal 001 vs 000)", "change_%": 100 * (n1 - n0) / n0},
            {"eff": eff, "model": m, "variation": "shower model (angular)",       "change_%": 100 * (ang - nom_avg) / nom_avg},
        ]
syst = pd.DataFrame(rows)
print(syst.round(2).to_string(index=False))
syst.to_csv(OUT + "robustness_corrected.csv", index=False)

## Plotting the data


### Plot 1 - The change in rejection per variation

In [ ]:
import matplotlib.pyplot as plt
sub = syst[syst.eff == 0.5]
pivot = sub.pivot(index="variation", columns="model", values="change_%")
pivot.plot.bar()
plt.axhline(0, color="black", lw=0.8)
plt.ylabel("Change in rejection at 50% eff. [%]")
plt.title("Sensitivity to systematic variations")
plt.tight_layout(); plt.savefig(OUT + "robustness_bars.png", dpi=150); plt.show()

### Plot 2 - accuracy vs robustness

The ideal model would sit at the bottom right, accurate and robust

In [ ]:
sub = syst[(syst.eff == 0.5) & ~syst.variation.str.startswith("control")]
nom50 = results[(results.eff == 0.5) & (results.variation == "nominal")].set_index("model")["rejection"]
worst = sub.groupby("model")["change_%"].apply(lambda c: c.abs().max())

plt.figure()
for m in nom50.index:
    plt.scatter(nom50[m], worst[m], s=120, label=m)
    plt.annotate(m, (nom50[m], worst[m]), textcoords="offset points", xytext=(8, 4))
plt.xlabel("Nominal rejection at 50% eff. (higher = more accurate)")
plt.ylabel("Largest |change| under variations [%] (lower = more robust)")
plt.legend(); plt.tight_layout()
plt.ylim(bottom=0)
plt.savefig(OUT + "accuracy_vs_robustness.png", dpi=150); plt.show()